In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline


Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [4]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.

定义一个名为NeuralNetwork的类，这是构建神经网络的核心部分。

In [7]:
class NeuralNetwork(nn.Module):#pytorch中，所有的神经网络模型都必须继承nn.Module。提供了各种功能。
    def __init__(self):
        super().__init__()#调用父类的初始化函数，这是必须的
        self.flatten = nn.Flatten()#定义一个展平层。它将 28x28 的二维图像输入转换为一个长度为 784 (28*28) 的一维向量，以便输入到全连接层中。
        self.linear_relu_stack = nn.Sequential(#这是一个 nn.Sequential 容器，它按顺序包装了多个模块
            nn.Linear(28*28, 512),#第一个全连接层（线性层），将 784 个输入特征转换为 512 个隐藏单元
            nn.ReLU(),#修正线性单元激活函数，引入非线性，使网络能够学习复杂的模式
            nn.Linear(512, 512),#第二个全连接层，输入和输出都是 512 个单元
            nn.ReLU(),
            nn.Linear(512, 10),#最后一层全连接层，将 512 个特征映射到 10 个输出单元（对应 FashionMNIST 的 10 个类别）
        )

    def forward(self, x):#forward 方法定义了数据在网络中流动的路径
        x = self.flatten(x)#首先通过展平层
        logits = self.linear_relu_stack(x)#然后通过线性层和激活函数的堆栈，得到最终的未归一化得分（Logits）
        return logits
        #应当通过调用 model(x) 来使用模型，而不是直接调用 model.forward(x)

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [8]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [9]:
X = torch.rand(1, 28, 28, device=device)
#创建一个随机张量来模拟输入数据。
#1 表示批量大小 (batch size)，即一次处理 1 张图。
#28, 28 表示图像的宽高。
#device=device 确保数据与模型位于同一设备（如 CPU 或 GPU）上。
#torch.manual_seed(25)  # 固定随机种子
#X = torch.rand(1, 28, 28, device=device)
#固定种子
logits = model(X)
#将输入 X 传给模型。这会自动调用模型中的 forward 方法。
#logits 是网络最后一层的输出，它是未归一化的原始分值。对于 10 分类任务，它的形状是 [1, 10]
pred_probab = nn.Softmax(dim=1)(logits)
#使用 Softmax 函数将 logits 转换为概率分布。
#dim=1 表示在类别维度上进行计算，使得这 10 个类别的概率之和等于 1。
y_pred = pred_probab.argmax(1)
#argmax(1) 找到概率最大的那个索引。
#这个索引就代表了模型预测的类别标签。
print(f"Predicted class: {y_pred}")

Predicted class: tensor([3])


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [17]:
import torch
# 设置打印选项，让张量完整显示不截断
torch.set_printoptions(threshold=float('inf'))
torch.manual_seed(25)
input_image = torch.rand(3, 5, 5)
print("--- 完整的输入张量 (3, 5, 5) ---")
print(input_image)
print(f"\nSize: {input_image.size()}")

--- 完整的输入张量 (3, 5, 5) ---
tensor([[[0.7518, 0.1929, 0.0629, 0.9118, 0.3828],
         [0.2990, 0.5933, 0.2911, 0.2416, 0.5582],
         [0.0481, 0.3497, 0.3520, 0.9528, 0.0284],
         [0.8488, 0.3947, 0.5181, 0.9726, 0.8813],
         [0.0056, 0.3056, 0.9384, 0.7949, 0.4399]],

        [[0.1766, 0.8739, 0.1425, 0.4682, 0.6254],
         [0.3040, 0.7923, 0.4691, 0.6875, 0.9917],
         [0.2772, 0.7970, 0.2249, 0.1119, 0.6863],
         [0.2238, 0.2678, 0.2246, 0.4711, 0.0603],
         [0.2517, 0.3705, 0.7340, 0.6466, 0.5172]],

        [[0.1176, 0.7000, 0.8191, 0.0488, 0.3021],
         [0.2490, 0.7769, 0.7847, 0.8554, 0.8310],
         [0.1154, 0.2578, 0.4702, 0.0530, 0.4207],
         [0.7639, 0.7536, 0.6063, 0.1899, 0.2837],
         [0.6097, 0.5808, 0.1660, 0.5746, 0.7927]]])

Size: torch.Size([3, 5, 5])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [18]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print("--- 完整的展平后张量 ---")
print(flat_image)
print(f"\nSize: {flat_image.size()}")

--- 完整的展平后张量 ---
tensor([[0.7518, 0.1929, 0.0629, 0.9118, 0.3828, 0.2990, 0.5933, 0.2911, 0.2416,
         0.5582, 0.0481, 0.3497, 0.3520, 0.9528, 0.0284, 0.8488, 0.3947, 0.5181,
         0.9726, 0.8813, 0.0056, 0.3056, 0.9384, 0.7949, 0.4399],
        [0.1766, 0.8739, 0.1425, 0.4682, 0.6254, 0.3040, 0.7923, 0.4691, 0.6875,
         0.9917, 0.2772, 0.7970, 0.2249, 0.1119, 0.6863, 0.2238, 0.2678, 0.2246,
         0.4711, 0.0603, 0.2517, 0.3705, 0.7340, 0.6466, 0.5172],
        [0.1176, 0.7000, 0.8191, 0.0488, 0.3021, 0.2490, 0.7769, 0.7847, 0.8554,
         0.8310, 0.1154, 0.2578, 0.4702, 0.0530, 0.4207, 0.7639, 0.7536, 0.6063,
         0.1899, 0.2837, 0.6097, 0.5808, 0.1660, 0.5746, 0.7927]])

Size: torch.Size([3, 25])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [24]:
layer1 = nn.Linear(in_features=5*5, out_features=25)
hidden1 = layer1(flat_image)
print("--- 完整的线性化后张量 ---")
print(hidden1)
print(f"\nSize: {hidden1.size()}")

--- 完整的线性化后张量 ---
tensor([[-0.1391, -0.4712, -0.2692, -0.1226,  0.5819, -0.2257,  0.2061, -0.3041,
          0.3761,  0.2521, -0.1232,  0.1331,  0.4661, -0.4052, -0.2125, -0.0268,
          0.5167, -0.4397,  0.2514,  0.3179, -0.4037, -0.4655,  0.4728,  0.0778,
         -0.0288],
        [-0.1427, -0.2332,  0.0712, -0.0110,  0.3380, -0.2742,  0.1110,  0.1790,
          0.1872, -0.1839,  0.1054,  0.0489,  0.4011, -0.4671, -0.6655, -0.3399,
          0.2661, -0.5040,  0.0870,  0.0383, -0.3718,  0.1510,  0.3292, -0.0461,
          0.1796],
        [ 0.3191, -0.2530, -0.1515, -0.1586,  0.1279, -0.1122,  0.1559,  0.3838,
          0.3117, -0.0864,  0.3395,  0.2604,  0.3859, -0.5279, -0.3458, -0.2470,
          0.5425, -0.4672,  0.3159, -0.3053, -0.4374, -0.1102,  0.3149,  0.1484,
         -0.0235]], grad_fn=<AddmmBackward0>)

Size: torch.Size([3, 25])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [25]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1391, -0.4712, -0.2692, -0.1226,  0.5819, -0.2257,  0.2061, -0.3041,
          0.3761,  0.2521, -0.1232,  0.1331,  0.4661, -0.4052, -0.2125, -0.0268,
          0.5167, -0.4397,  0.2514,  0.3179, -0.4037, -0.4655,  0.4728,  0.0778,
         -0.0288],
        [-0.1427, -0.2332,  0.0712, -0.0110,  0.3380, -0.2742,  0.1110,  0.1790,
          0.1872, -0.1839,  0.1054,  0.0489,  0.4011, -0.4671, -0.6655, -0.3399,
          0.2661, -0.5040,  0.0870,  0.0383, -0.3718,  0.1510,  0.3292, -0.0461,
          0.1796],
        [ 0.3191, -0.2530, -0.1515, -0.1586,  0.1279, -0.1122,  0.1559,  0.3838,
          0.3117, -0.0864,  0.3395,  0.2604,  0.3859, -0.5279, -0.3458, -0.2470,
          0.5425, -0.4672,  0.3159, -0.3053, -0.4374, -0.1102,  0.3149,  0.1484,
         -0.0235]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.5819, 0.0000, 0.2061, 0.0000, 0.3761,
         0.2521, 0.0000, 0.1331, 0.4661, 0.0000, 0.0000, 0.0000, 0.5167, 0.0000

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [32]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(25, 5) # 输入维度必须匹配前一层的输出维度25，这里输出维度设为10
)
input_image = torch.rand(3, 5, 5)
logits = seq_modules(input_image)
print(f"Logits size: {logits.size()}")
print(logits)

Logits size: torch.Size([3, 5])
tensor([[-0.0241,  0.0472,  0.1110,  0.0387,  0.1484],
        [ 0.0110,  0.0454,  0.0800,  0.0849,  0.2685],
        [ 0.0734,  0.1307,  0.1810,  0.0477,  0.3206]],
       grad_fn=<AddmmBackward0>)


nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.

线性返回曾=层返回结果，进行归一化处理并输出预测概率

In [35]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.1828, 0.1963, 0.2092, 0.1946, 0.2172],
        [0.1826, 0.1890, 0.1956, 0.1966, 0.2362],
        [0.1842, 0.1951, 0.2052, 0.1796, 0.2359]], grad_fn=<SoftmaxBackward0>)


Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [36]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-1.0121e-04,  3.0402e-02, -2.8198e-02,  3.0648e-02, -3.1148e-02,
         -3.4035e-02, -3.3122e-02, -2.3448e-02, -1.1097e-04,  7.6041e-03,
         -2.2381e-02,  4.0937e-04,  2.7812e-02, -7.1280e-03, -1.4393e-03,
         -2.6906e-02,  1.9458e-02, -3.3760e-02,  2.3803e-02,  3.3542e-02,
          1.8017e-02,  1.5182e-02, -1.1920e-04, -3.1230e-02, -2.2431e-02,
         -3.1723e-02,  7.9464e-03,  4.1771e-04, -2.1085e-02, -2.2106e-02,
          3.3294e-02, -1.6124e-02, -3.7603e-03, -1.8766e-02,  1.6183e-02,
          2.4212e-02, -9.8767e-03, -1.3470e-02,  3.8500e-03,  2.1560

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
